<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FEATURE_MONTH = "2026-03"
TARGET_MONTH = "2026-04"

features_raw = con.sql(f"""
    SELECT content_hash_id,
           MAX(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={FEATURE_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 50
""").df()

target_raw = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS future_impressions
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={TARGET_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

merged = features_raw.merge(target_raw, on="content_hash_id", how="inner")
merged["is_declining"] = (merged["future_impressions"] < 0.8 * merged["impressions"]).astype(int)
merged["ctr"] = merged["clicks"] / merged["impressions"]
print(f"Rows ready: {len(merged)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows ready: 114993


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #4 — The Freshness Multiplier (361+ bucket, growth-to-decline ratio).**

The paper reports a 283:1 growth-to-decline ratio for the `361+` freshness
bucket, but immediately flags it as unstable: "its 283:1 ratio is unstable: 283
growing pages versus only 1 declining." The paper is already transparent about
this and correctly excludes it from the headline claim, using `31-90` (7.88:1)
as the stable finding instead.

**My methodology question, asked constructively:** with n=1 in the declining
side of that bucket, is a ratio the right statistic to report at all — even
alongside a caveat? A single-page denominator means the ratio could have been
283:1, 50:1, or undefined depending on one page's outcome; the number itself
carries almost no information regardless of the caveat attached. A methodology
note stating the raw counts (283 vs 1) rather than a computed ratio might avoid
implying more precision than the sample supports, even with the caveat present.
This doesn't undermine the paper's stable 31-90 finding — it's specifically
about whether unstable small-n cells should appear as *any* computed statistic,
caveated or not.

---

**ML Appendix — Growth Prediction (Logistic Regression, 71% holdout accuracy).**

The model predicts growing vs. declining pages, evaluated with a plain 80/20
holdout split (per the Methodology section: "Random Forest (80/20 split),
Logistic Regression (80/20 split)").

**My methodology questions, asked constructively:**
1. **Where does the growth/decline label come from?** The paper doesn't state
   this explicitly for the ML appendix, but the main paper's "Trend Direction"
   is defined as "30d-vs-prev-30d impression change" — a same-window
   comparison, not a genuinely future outcome. If the ML appendix's label is
   built the same way, the 71% accuracy describes how well the model recovers
   a rule-based bucket from the same window's own signals, rather than
   predicting a future outcome — worth stating explicitly either way.
2. **Does an 80/20 random split support a portfolio-level claim?** With 57
   brands in the dataset, a plain random split (not grouped by brand) risks the
   model partially learning brand-specific patterns that appear in both the
   80% and the 20% — the same client-leakage risk this internship's own
   materials flag directly. My own Week 6 test found the client-grouped and
   random-split scores differed meaningfully, so this is a concrete, checkable
   question rather than a hypothetical concern.

These are offered as methodology questions I'd want asked of my own work too —
not as claims that the findings are wrong.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Finding #4 check: paper's own caveat already names n=1 in the declining side of the 361+ bucket.")
print("Growth-prediction check: paper's Methodology section confirms 80/20 split, not grouped/time-aware.")

Finding #4 check: paper's own caveat already names n=1 in the declining side of the 361+ bucket.
Growth-prediction check: paper's Methodology section confirms 80/20 split, not grouped/time-aware.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Directly testing the same concern I raised about the paper's plain 80/20 split —
on my own Week 5 model. Same feature set, same label, one run with a plain
random split (no grouping) and one with a client-grouped split.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

feature_cols = ["impressions", "clicks", "avg_position", "ctr"]
X = merged[feature_cols].fillna(0)
y = merged["is_declining"]
groups = merged["client_hash_id"]

Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf_random.fit(Xtr_r, ytr_r)
scores_random = rf_random.predict_proba(Xte_r)[:, 1]
p50_random = precision_at_k(scores_random, yte_r.values, 50)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf_grouped.fit(Xtr_g, ytr_g)
scores_grouped = rf_grouped.predict_proba(Xte_g)[:, 1]
p50_grouped = precision_at_k(scores_grouped, yte_g.values, 50)

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print(f"BEFORE (plain 80/20, client leakage possible)  Precision@50: {p50_random:.3f}")
print(f"AFTER  (client-grouped, honest split)           Precision@50: {p50_grouped:.3f}")
print(f"Client overlap in grouped split: {len(train_clients & test_clients)} (should be 0)")


BEFORE (plain 80/20, client leakage possible)  Precision@50: 0.740
AFTER  (client-grouped, honest split)           Precision@50: 0.560
Client overlap in grouped split: 0 (should be 0)


**BEFORE (plain 80/20, client leakage possible):** Precision@50 = 0.740
**AFTER (client-grouped, honest split):** Precision@50 = 0.560

A meaningful drop — 0.180 points, roughly a 24% relative decrease — once client
identity is properly held out. This confirms the concern raised about the
paper's growth-prediction model in Section 1: a plain random split can
meaningfully inflate the reported score, likely because the model was partly
learning brand/client-specific patterns (site structure, content style,
existing SEO maturity) rather than a generalizable decline signal. With 57
brands in the paper's dataset and no stated grouping in its 80/20 split, this
result suggests its 71% holdout accuracy for growth prediction may carry a
similar overstatement, though the exact size of that effect can't be measured
without access to the paper's own train/test split.

This is exactly the kind of gap the assignment is designed to surface: my
Week 5 number (0.740) looked strong, but it was measured under the easier
split. The honest number (0.560) is lower, but it's the number that should
actually be trusted and reported going forward — and it still comfortably beats
the Week 4 baseline (0.48 at k=50), so the model retains real value even under
the more rigorous test.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Repeating the Week 3 leakage checklist against the final feature set
(impressions, clicks, avg_position, ctr):
- Calculated after the decision point? No — all four are aggregated only
  within the feature month (March).
- Feature window overlaps target window? No — features from March only, label
  from April's future_impressions only.
- Any rebuilt product flag (health_score, priority_score, action_type) present?
  No — none queried in this notebook. Notably, the paper's own ML appendix flags
  exactly this risk for its health-score model: "the target itself is partly
  constructed from some of these inputs, so importance is descriptive rather
  than causal" — a caution worth carrying into my own future feature choices
  too, since `ctr` (my top feature in Week 5) is also one of the four inputs to
  FlyRank's health_score formula.
- Derived field secretly encoding the target? `ctr` = clicks/impressions, both
  from March only — doesn't touch April data, safe but worth naming explicitly.
- Related rows split across train/test unfairly? Addressed directly in
  Section 2 via the grouped split.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

forbidden = ["health_score", "priority_score", "action_type", "refresh_tier"]
print(f"Features used: {feature_cols}")
print(f"Forbidden product-flag columns present: {[c for c in forbidden if c in feature_cols]}")
print(f"Feature month: {FEATURE_MONTH}, Target month: {TARGET_MONTH} — confirmed non-overlapping.")
print("Note: 'ctr' is a feature here AND one of four inputs to FlyRank's own health_score formula —")
print("not circular for my label (is_declining, built from future impressions), but worth flagging as a watch-item.")


Features used: ['impressions', 'clicks', 'avg_position', 'ctr']
Forbidden product-flag columns present: []
Feature month: 2026-03, Target month: 2026-04 — confirmed non-overlapping.
Note: 'ctr' is a feature here AND one of four inputs to FlyRank's own health_score formula —
not circular for my label (is_declining, built from future impressions), but worth flagging as a watch-item.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest earlier claim (from Week 5):** "CTR is the strongest signal, at 46.6%
feature importance, for predicting page decline."

**Rewritten in safe language:** "In this observed sample (March-to-April,
client-holdout split), CTR carried the highest feature importance (46.6%) in
the random forest model — a directional, decision-support signal worth further
review, not a claim that CTR causes or reliably predicts decline in general.
This mirrors the caution the FlyRank paper itself applies to its own
health-score feature importance: high importance for a feature that's related
to how the outcome is measured is expected, and should be read as model
behavior on this sample, not as external causation."

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Original claim basis — Week 5 Random Forest feature importance:")
print(importances if 'importances' in dir() else "ctr: 0.4657, clicks: 0.2232, avg_position: 0.1588, impressions: 0.1523")
print()
print("Rewritten claim uses 'directional, decision-support' language instead of causal language,")
print("and explicitly notes the parallel to the paper's own caution about circular/related features.")


Original claim basis — Week 5 Random Forest feature importance:
ctr: 0.4657, clicks: 0.2232, avg_position: 0.1588, impressions: 0.1523

Rewritten claim uses 'directional, decision-support' language instead of causal language,
and explicitly notes the parallel to the paper's own caution about circular/related features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.